## 1. Импорт библиотек и настройка запроса



In [10]:
import re
import time
import json
from pathlib import Path
from urllib.parse import urljoin, urlparse
from collections import OrderedDict

import pandas as pd
import requests
from bs4 import BeautifulSoup
from IPython.display import display

BASE_URL = "https://lifehacker.ru/topics/technology/"
PAGES_COUNT = 10
REQUEST_DELAY = 1.0
TIMEOUT = 20

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Accept-Language": "ru-RU,ru;q=0.9,en;q=0.8",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
})

print("Базовая ссылка:", BASE_URL)


Базовая ссылка: https://lifehacker.ru/topics/technology/


## 2. Формат пагинации



In [11]:
def page_url(page_number: int) -> str:
    if not 1 <= page_number <= PAGES_COUNT:
        raise ValueError("Номер страницы должен быть от 1 до 10")
    return BASE_URL if page_number == 1 else f"{BASE_URL}?page={page_number}"

page_urls = [page_url(n) for n in range(1, PAGES_COUNT + 1)]
print("Первая страница:", page_urls[0])
print("Вторая страница:", page_urls[1])
print("Десятая страница:", page_urls[-1])
assert len(page_urls) == 10 and page_urls[-1].endswith("?page=10")

Первая страница: https://lifehacker.ru/topics/technology/
Вторая страница: https://lifehacker.ru/topics/technology/?page=2
Десятая страница: https://lifehacker.ru/topics/technology/?page=10


## 3. Определение текущей разметки и выделение ссылок




In [12]:
def clean_text(value):
    return re.sub(r"\s+", " ", value or "").strip()


def normalise_article_url(href, base_url=BASE_URL):
    if not isinstance(href, str) or not href.strip():
        return None
    u = urlparse(urljoin(base_url, href.strip()))
    if u.scheme not in ("http", "https") or u.hostname not in ("lifehacker.ru", "www.lifehacker.ru"):
        return None
    slug = u.path.strip("/")
    # Публикации Lifehacker обычно доступны по /slug/. Служебные пути исключаем.
    if not re.fullmatch(r"[\w-]{3,}", slug, re.UNICODE):
        return None
    if slug.lower() in {"topics", "technology", "tags", "tag", "authors", "author",
                        "news", "page", "search", "login", "feed", "about", "list",
                        "contacts", "advertising", "privacy", "rss", "best", "services"}:
        return None
    return "https://lifehacker.ru/" + slug + "/"


def find_card_title(a):
    # Варианты разметки: a > h2; h2 > a; a[aria-label]; a > img[alt];
    # a находится рядом с заголовком внутри родительской карточки.
    candidates = []
    h = a.find(["h2", "h3", "h4"])
    if h:
        candidates.append(h.get_text(" ", strip=True))
    if a.parent and a.parent.name in ("h2", "h3", "h4"):
        candidates.append(a.parent.get_text(" ", strip=True))
    candidates.extend([a.get_text(" ", strip=True), a.get("aria-label", ""), a.get("title", "")])
    img = a.find("img")
    if img:
        candidates.append(img.get("alt", ""))
    node = a
    for _ in range(3):
        node = node.parent
        if not node or node.name in ("main", "body", "html"):
            break
        heading = node.find(["h2", "h3", "h4"])
        if heading:
            candidates.append(heading.get_text(" ", strip=True))
    return next((clean_text(t) for t in candidates if len(clean_text(t)) >= 12), "")


def extract_article_links(html, base_url=BASE_URL):
    soup = BeautifulSoup(html, "html.parser")
    result = OrderedDict()
    # Не ограничиваем поиск первым main: некоторые шаблоны используют несколько секций.
    for a in soup.select("a[href]"):
        if a.find_parent(["nav", "footer", "header", "aside"]):
            continue
        # Не исключаем целые родительские блоки по regex классов: иногда это удаляет всю ленту.
        url = normalise_article_url(a.get("href"), base_url)
        if not url:
            continue
        title = find_card_title(a)
        if not title:
            continue
        result.setdefault(url, {"url": url, "title_in_list": title})

    # Дополнительный путь: Schema.org ItemList / Article в JSON-LD внутри исходного HTML.
    # Применяется, если прямых карточек нет; не требуется JavaScript или обход блокировок.
    if not result:
        def walk(data):
            if isinstance(data, list):
                for el in data:
                    yield from walk(el)
            elif isinstance(data, dict):
                yield data
                for el in data.values():
                    yield from walk(el)
        for script in soup.select('script[type="application/ld+json"]'):
            try:
                data = json.loads(script.string or script.get_text())
            except (TypeError, ValueError):
                continue
            for obj in walk(data):
                if not isinstance(obj, dict):
                    continue
                raw_url = obj.get("url") or obj.get("@id")
                if isinstance(raw_url, dict):
                    raw_url = raw_url.get("@id")
                title = obj.get("headline") or obj.get("name")
                url = normalise_article_url(raw_url, base_url)
                if url and isinstance(title, str) and len(clean_text(title)) >= 12:
                    result.setdefault(url, {"url": url, "title_in_list": clean_text(title)})
    return list(result.values())


## 4. Диагностика первой страницы и сбор списка статей




In [13]:
def download_html(url):
    r = session.get(url, timeout=TIMEOUT)
    r.raise_for_status()
    if "text/html" not in r.headers.get("Content-Type", "text/html"):
        raise ValueError("Ответ сервера не является HTML: " + r.headers.get("Content-Type", "?"))
    return r.content


def html_diagnostics(html):
    soup = BeautifulSoup(html, "html.parser")
    title = soup.title.get_text(" ", strip=True) if soup.title else "(не найден)"
    anchors = soup.select("a[href]")
    normalized = [(a.get("href", ""), normalise_article_url(a.get("href"))) for a in anchors]
    candidates = [(raw, url) for raw, url in normalized if url]
    print("Размер HTML:", len(html), "байт")
    print("Заголовок <title>:", title)
    print("Тегов <main>:", len(soup.select("main")), "; ссылок <a href>:", len(anchors))
    print("Ссылок с возможным адресом статьи:", len(candidates))
    print("Примеры <a href>:")
    for a in anchors[:10]:
        print("  ", a.get("href", "")[:120], "|", clean_text(a.get_text(" ", strip=True))[:65])
    print("Примеры адресов статей:")
    for raw, url in candidates[:6]:
        print("  ", url)
    if re.search(r"captcha|just a moment|access denied|verify you are human|cloudflare", title, re.I):
        print("ВНИМАНИЕ: сервер, возможно, возвратил проверку доступа вместо рубрики.")

parsed_items = OrderedDict()
first_page_html = None
page_report = []
failed_pages = []

for page_num, url in enumerate(page_urls, 1):
    try:
        html = download_html(url)
        if page_num == 1:
            first_page_html = html
            print("Диагностика первой страницы:")
            html_diagnostics(html)
            # Файл поможет определить актуальную разметку; полагаться на DOM браузера нельзя.
            Path("lifehacker_page1_debug.html").write_bytes(html)
        cards = extract_article_links(html, base_url=url)
        page_report.append({"page": page_num, "url": url, "links_found": len(cards)})
        for card in cards:
            parsed_items.setdefault(card["url"], {**card, "page": page_num})
        print(f"Страница {page_num:2d}/10: найдено {len(cards)} ссылок")
        if not cards:
            failed_pages.append({"page": page_num, "url": url, "error": "В HTML не найдены карточки/ссылки"})
    except (requests.RequestException, ValueError) as exc:
        print(f"Страница {page_num:2d}/10: ошибка — {exc}")
        page_report.append({"page": page_num, "url": url, "links_found": 0})
        failed_pages.append({"page": page_num, "url": url, "error": str(exc)})
    time.sleep(REQUEST_DELAY)

links_df = pd.DataFrame(parsed_items.values(), columns=["url", "title_in_list", "page"])
print("Всего уникальных ссылок:", len(links_df))
display(pd.DataFrame(page_report))
display(links_df.head())
if not parsed_items:
    print("НЕ ПРОДОЛЖАЙТЕ СДАЧУ ПУСТОГО РЕЗУЛЬТАТА. Проверьте lifehacker_page1_debug.html и вывод диагностики.")


Диагностика первой страницы:
Размер HTML: 826353 байт
Заголовок <title>: Технологии — Лайфхакер
Тегов <main>: 1 ; ссылок <a href>: 170
Ссылок с возможным адресом статьи: 82
Примеры <a href>:
   / | 
   / | 
   /top/week/ | Лучшее
   /topics/news/ | Новости
   /topics/life/ | Жизнь
   /recipes/ | Рецепты
   /health/ | Здоровье
   /topics/relax/movies/ | Кино
   /topics/shopping/ | Покупки
   /top/week/ | Лучшее
Примеры адресов статей:
   https://lifehacker.ru/recipes/
   https://lifehacker.ru/health/
   https://lifehacker.ru/recipes/
   https://lifehacker.ru/health/
   https://lifehacker.ru/v-android-nashli-uyazvimost/
   https://lifehacker.ru/v-android-nashli-uyazvimost/
Страница  1/10: найдено 30 ссылок
Страница  2/10: найдено 30 ссылок
Страница  3/10: найдено 30 ссылок
Страница  4/10: найдено 30 ссылок
Страница  5/10: найдено 30 ссылок
Страница  6/10: найдено 30 ссылок
Страница  7/10: найдено 30 ссылок
Страница  8/10: найдено 30 ссылок
Страница  9/10: найдено 30 ссылок
Страница 10/10

,page,url,links_found
0,1,https://lifehacker.ru/topics/technology/,30
1,2,https://lifehacker.ru/topics/technology/?page=2,30
2,3,https://lifehacker.ru/topics/technology/?page=3,30
3,4,https://lifehacker.ru/topics/technology/?page=4,30
4,5,https://lifehacker.ru/topics/technology/?page=5,30
5,6,https://lifehacker.ru/topics/technology/?page=6,30
6,7,https://lifehacker.ru/topics/technology/?page=7,30
7,8,https://lifehacker.ru/topics/technology/?page=8,30
8,9,https://lifehacker.ru/topics/technology/?page=9,30
9,10,https://lifehacker.ru/topics/technology/?page=10,30


,url,title_in_list,page
0,https://lifehacker.ru/v-android-nashli-uyazvimost/,В смартфонах 7&nbsp;производителей нашли брешь для перехвата данных,1
1,https://lifehacker.ru/vyxod-magicos-11/,"Названы устройства Honor, которые получат MagicOS 11 на базе Android 17",1
2,https://lifehacker.ru/luchshie-izobreteniya-2026-goda-time/,Журнал Time назвал 300 лучших изобретений 2026 года. Выбрали 15 самых интересных,1
3,https://lifehacker.ru/chatgpt-nauchilsya-prosmatrivat-i-upravlyat-vkladkami-chrome/,ChatGPT научился просматривать и&nbsp;управлять вкладками Chrome — без лишних приложений,1
4,https://lifehacker.ru/chto-luchshe-oled-ili-mini-led/,Что лучше — OLED или mini-LED,1


### Проверка реальных классов/ID блока статьи 




In [14]:
if first_page_html and parsed_items:
    soup = BeautifulSoup(first_page_html, "html.parser")
    sample_url = next(iter(parsed_items))
    anchor = next((a for a in soup.select("a[href]")
                   if normalise_article_url(a.get("href")) == sample_url), None)
    if anchor:
        rows = []
        for i, node in enumerate([anchor, *list(anchor.parents)[:5]]):
            if node.name == "[document]":
                continue
            rows.append({"глубина": i, "тег": node.name,
                         "class": " ".join(node.get("class", [])), "id": node.get("id", "")})
        print("Пример:", sample_url)
        display(pd.DataFrame(rows))
    else:
        print("Статьи извлечены из JSON-LD, прямой <a> не найден. Изучите HTML диагностической страницы.")
else:
    print("Классы не определены: на первой странице не обнаружены материалы.")


Пример: https://lifehacker.ru/v-android-nashli-uyazvimost/


,глубина,тег,class,id
0,0,a,lh-small-article-card__link,
1,1,div,lh-small-article-card article-card__small,2231209
2,2,div,article-card__small-wrapper,
3,3,div,container,
4,4,div,article-card-container,
5,5,div,archive-page,


## 5. Поиск HTML-блока с заголовком и текстом отдельного материала



In [15]:
ARTICLE_SELECTORS = [
    '[itemprop="articleBody"]',
    '[class*="article-content"]',
    '[class*="articleContent"]',
    '[class*="article__text"]',
    '[class*="article_text"]',
    '[class*="post__body"]',
    '[class*="entry-content"]',
    'article [class*="article__content"]',
    'article [class*="article-body"]',
    '[class*="post-content"]',
    '[class*="article__body"]',
    '[class*="article__content"]',
    '[class*="article-body"]',
    'article',
]


def extract_article_data(html: str, url: str) -> dict:
    soup = BeautifulSoup(html, "html.parser")
    h1 = soup.select_one("h1")
    title = clean_text(h1.get_text(" ", strip=True)) if h1 else ""
    if not title:
        meta = soup.select_one('meta[property="og:title"]')
        title = clean_text(meta.get("content", "")) if meta else ""

    body = None
    for selector in ARTICLE_SELECTORS:
        candidate = soup.select_one(selector)
        if candidate and len(candidate.select("p")) >= 1:
            body = candidate
            break
    if body is None:
        raise ValueError("Не найден HTML-блок текста статьи")

    # Создаём отдельную копию HTML, чтобы безопасно удалить не относящиеся к статье элементы.
    cleaned = BeautifulSoup(str(body), "html.parser")
    for el in cleaned.select("script, style, noscript, nav, aside, footer, header, form, button, iframe"):
        el.decompose()
    for el in list(cleaned.select('[class*="related"], [class*="social"], [class*="advert"], [class*="promo"], [class*="share"], [class*="banner"]')):
        el.decompose()

    blocks = []
    for el in cleaned.select("h2, h3, p, li"):
        # В списке могут находиться вложенные <p> — не дублируем их.
        if el.name == "li" and el.find(["p", "h2", "h3"]):
            continue
        block = clean_text(el.get_text(" ", strip=True))
        if block:
            blocks.append(block)
    text = "\n\n".join(blocks)
    if not title:
        raise ValueError("Не найден заголовок статьи")
    if not text:
        raise ValueError("Не найден текст статьи")
    return {"url": url, "title": title, "text": text, "text_length": len(text)}


## 6. Перебор ссылок на материалы



In [16]:
result = []
failed_articles = []
total = len(parsed_items)

for index, (url, card) in enumerate(parsed_items.items(), start=1):
    try:
        article_html = download_html(url)
        article = extract_article_data(article_html, url)
        article["page"] = card["page"]
        result.append(article)
    except (requests.RequestException, ValueError) as exc:
        failed_articles.append({"url": url, "page": card["page"], "error": str(exc)})
    if index % 10 == 0 or index == total:
        print(f"Обработано {index}/{total}; успешно: {len(result)}; ошибок: {len(failed_articles)}")
    time.sleep(REQUEST_DELAY)

print(f"Итог: {len(result)} статей с заголовком и текстом")

Обработано 10/300; успешно: 10; ошибок: 0
Обработано 20/300; успешно: 20; ошибок: 0
Обработано 30/300; успешно: 30; ошибок: 0
Обработано 40/300; успешно: 40; ошибок: 0
Обработано 50/300; успешно: 50; ошибок: 0
Обработано 60/300; успешно: 60; ошибок: 0
Обработано 70/300; успешно: 70; ошибок: 0
Обработано 80/300; успешно: 80; ошибок: 0
Обработано 90/300; успешно: 90; ошибок: 0
Обработано 100/300; успешно: 100; ошибок: 0
Обработано 110/300; успешно: 110; ошибок: 0
Обработано 120/300; успешно: 120; ошибок: 0
Обработано 130/300; успешно: 130; ошибок: 0
Обработано 140/300; успешно: 140; ошибок: 0
Обработано 150/300; успешно: 150; ошибок: 0
Обработано 160/300; успешно: 160; ошибок: 0
Обработано 170/300; успешно: 170; ошибок: 0
Обработано 180/300; успешно: 180; ошибок: 0
Обработано 190/300; успешно: 190; ошибок: 0
Обработано 200/300; успешно: 200; ошибок: 0
Обработано 210/300; успешно: 210; ошибок: 0
Обработано 220/300; успешно: 220; ошибок: 0
Обработано 230/300; успешно: 230; ошибок: 0
Обрабо

## 7. Создание DataFrame и проверка результата



In [17]:
data = pd.DataFrame(result, columns=["page", "url", "title", "text", "text_length"])
failed_articles_df = pd.DataFrame(failed_articles, columns=["url", "page", "error"])

pd.set_option("display.max_colwidth", 140)
print("Размер итоговой таблицы:", data.shape)
print("Всего страниц обработано:", len(page_report))
print("Уникальных ссылок:", len(parsed_items))
print("Успешно извлечённых материалов:", len(data))
print("Ошибок страниц:", len(failed_pages))
print("Ошибок статей:", len(failed_articles_df))
if not data.empty:
    print("Пустые значения по столбцам:")
    display(data.isna().sum().to_frame("missing"))
    display(data[["page", "url", "title", "text_length"]].head(10))
    display(data[["title", "text"]].head(3))
else:
    print("ВНИМАНИЕ: реальные данные не получены. Проверьте доступность сайта и HTML-разметку.")

if failed_pages:
    print("Проблемы на страницах рубрики:")
    display(pd.DataFrame(failed_pages))
if not failed_articles_df.empty:
    print("Примеры проблемных материалов:")
    display(failed_articles_df.head(10))

Размер итоговой таблицы: (300, 5)
Всего страниц обработано: 10
Уникальных ссылок: 300
Успешно извлечённых материалов: 300
Ошибок страниц: 0
Ошибок статей: 0
Пустые значения по столбцам:


,missing
page,0
url,0
title,0
text,0
text_length,0


,page,url,title,text_length
0,1,https://lifehacker.ru/v-android-nashli-uyazvimost/,В смартфонах 7 производителей смартфонов нашли брешь для перехвата данных,1677
1,1,https://lifehacker.ru/vyxod-magicos-11/,"Названы устройства Honor, которые получат MagicOS 11 на базе Android 17",2189
2,1,https://lifehacker.ru/luchshie-izobreteniya-2026-goda-time/,Журнал Time назвал 300 лучших изобретений 2026 года. Выбрали 15 самых интересных,7226
3,1,https://lifehacker.ru/chatgpt-nauchilsya-prosmatrivat-i-upravlyat-vkladkami-chrome/,ChatGPT научился просматривать и управлять вкладками Chrome — без лишних приложений,1934
4,1,https://lifehacker.ru/chto-luchshe-oled-ili-mini-led/,Что лучше — OLED или mini-LED,10546
5,1,https://lifehacker.ru/obzor-honor-x9e-pro/,Обзор Honor X9e Pro — сверхпрочного смартфона с батарейкой 11 000 мА·ч,12811
6,1,https://lifehacker.ru/eksperty-prizvali-zaperet-neiroseti/,«Крёстные отцы ИИ» и 400 учёных потребовали запереть мощные нейросети в «песочницах»,1930
7,1,https://lifehacker.ru/anons-ledger-atom/,"Ledger представила Android-смартфон, с которого нельзя звонить. Для чего он нужен?",3597
8,1,https://lifehacker.ru/anons-red-magic-12-pro/,ZTE выпустила монструозный игровой смартфон с батареей на 8 400 мА⋅ч и экраном с частотой 185 Гц,2029
9,1,https://lifehacker.ru/kak-proverit-skorost-interneta-na-kompyutere-i-telefone/,Как проверить скорость интернета на компьютере и телефоне,12611


,title,text
0,В смартфонах 7 производителей смартфонов нашли брешь для перехвата данных,Автор Лайфхакера\n\nИсследователи Грацского технического университета и австрийского Центра безопасных информационных технологий обнаруж...
1,"Названы устройства Honor, которые получат MagicOS 11 на базе Android 17",Автор Лайфхакера\n\nHonor раскрыла график глобального релиза MagicOS 11 на базе Android 17 — с дизайном «жидкого стекла» и расширенными ...
2,Журнал Time назвал 300 лучших изобретений 2026 года. Выбрали 15 самых интересных,"Автор Лайфхакера\n\nАмериканский журнал Time опубликовал ежегодный список лучших изобретений. В него вошли 300 устройств, технологий и с..."


## 8. Сохранение результата



In [18]:
from pathlib import Path
# Записываем результаты внутрь hw_7, независимо от того, где VS Code создал рабочую папку.
workdir = Path.cwd()
output_dir = workdir / "hw_7" if (workdir / "hw_7" / "hw_7.ipynb").exists() else workdir
OUTPUT_CSV = output_dir / "lifehacker_technology_10_pages.csv"
ERRORS_CSV = output_dir / "lifehacker_scraping_errors.csv"
data.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
failed_articles_df.to_csv(ERRORS_CSV, index=False, encoding="utf-8-sig")
print("Сохранено:", OUTPUT_CSV)
print("Журнал ошибок:", ERRORS_CSV)

if failed_pages or failed_articles or data.empty:
    print("ВНИМАНИЕ: сбор неполный. Проверьте журнал и повторите неудачные запросы, не обходя блокировок.")
else:
    print("Готово: все 10 страниц обработаны, а тексты найденных статей извлечены.")

Сохранено: c:\Users\roman\Desktop\python-homework\hw_7\lifehacker_technology_10_pages.csv
Журнал ошибок: c:\Users\roman\Desktop\python-homework\hw_7\lifehacker_scraping_errors.csv
Готово: все 10 страниц обработаны, а тексты найденных статей извлечены.
